# Инициализация

Загружаем библиотеки необходимые для выполнения кода ноутбука.

In [1]:
import os
import boto3
from dotenv import load_dotenv
import sys
import scipy.sparse
from IPython.display import display
import gc
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


load_dotenv()

True

In [2]:
import utils
import importlib

importlib.reload(utils)

<module 'utils' from '/home/mle-user/sprint5/final_project/utils.py'>

In [3]:
import boto3, os
s3 = boto3.client(
    "s3",
    endpoint_url="https://storage.yandexcloud.net",
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
)
bucket_name = os.getenv("S3_BUCKET_NAME")

In [4]:
objects = []

paginator = s3.get_paginator("list_objects_v2")

for page in paginator.paginate(Bucket=bucket_name):
    for obj in page.get("Contents", []):
        objects.append({
            "key": obj["Key"],
            "size_mb": obj["Size"] / 1024**2,
            "last_modified": obj["LastModified"],
        })

objects_sorted = sorted(
    objects,
    key=lambda x: x["size_mb"],
    reverse=True
)

for obj in objects_sorted[:50]:
    print(
        f'{obj["size_mb"]:10.2f} MB | '
        f'{obj["last_modified"]} | '
        f'{obj["key"]}'
    )

   2515.25 MB | 2026-09-30 07:58:04.072000+00:00 | recsys/candidates/candidates_result.parquet
   1159.84 MB | 2026-09-30 08:15:14.095000+00:00 | recsys/recommendations/recommendations.parquet
    710.62 MB | 2026-09-29 16:20:39.430000+00:00 | recsys/recommendations/personal_als.parquet
    342.75 MB | 2026-09-29 14:36:35.127000+00:00 | recsys/data/item_properties.parquet
     84.51 MB | 2026-09-29 16:20:40.324000+00:00 | recsys/recommendations/similar.parquet
     39.95 MB | 2026-09-29 14:36:33.075000+00:00 | recsys/data/events.parquet
     18.20 MB | 2026-06-11 14:12:22.639000+00:00 | files/md5/eb/9344ea0dd74cdcb5513f6de662a828
     15.52 MB | 2026-06-14 10:27:58.965000+00:00 | files/md5/d6/2e824fe2d6bfa01603e58b274094b9
     11.49 MB | 2026-08-13 14:33:24.846000+00:00 | models/step5.csv
      2.23 MB | 2026-08-13 14:33:23.975000+00:00 | models/fitted_model.pkl
      1.04 MB | 2026-06-14 10:27:58.657000+00:00 | files/md5/c8/56cafef77d44be13a67efb964d27b5
      1.04 MB | 2026-06-16 17

# Ранжирование рекомендаций

Построим ранжирующую модель, чтобы сделать рекомендации более точными. Отранжируем рекомендации.

In [5]:
# ============================================================
# ПОДГОТОВКА ДАННЫХ ДЛЯ РАНЖИРУЮЩЕЙ МОДЕЛИ
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Загружаем кандидатов с уже построенными признаками
# ------------------------------------------------------------

candidates = pd.read_parquet(
    "candidates_result.parquet"
)


# ------------------------------------------------------------
# 2. Подготавливаем events
# ------------------------------------------------------------

events = pd.read_parquet(
    "parquet_cleaned/events.parquet"
)


# Если названия ещё не приведены к общему виду
events_rank = events.rename(
    columns={
        "visitorid": "user_id",
        "itemid": "item_id",
    }
)


# ------------------------------------------------------------
# 3. Приводим timestamp к datetime
# ------------------------------------------------------------

if pd.api.types.is_numeric_dtype(
    events_rank["timestamp"]
):
    events_rank["event_time"] = pd.to_datetime(
        events_rank["timestamp"],
        unit="ms"
    )
else:
    events_rank["event_time"] = pd.to_datetime(
        events_rank["timestamp"]
    )


print(
    "Период данных:",
    events_rank["event_time"].min(),
    "->",
    events_rank["event_time"].max()
)

Период данных: 2015-05-03 03:00:04.384000 -> 2015-09-18 02:59:47.788000


In [6]:
# ============================================================
# ВРЕМЕННОЙ SPLIT ДЛЯ RANKER
# ============================================================

data_end = (
    events_rank["event_time"].max()
    + pd.Timedelta(milliseconds=1)
)

# Последние 7 дней -> финальная оценка
eval_start = (
    data_end
    - pd.Timedelta(days=7)
)

# Предыдущие 7 дней -> обучение ranker
ranker_start = (
    eval_start
    - pd.Timedelta(days=7)
)


print(
    "История для построения рекомендаций:",
    events_rank["event_time"].min(),
    "->",
    ranker_start
)

print(
    "Ranker train:",
    ranker_start,
    "->",
    eval_start
)

print(
    "Evaluation:",
    eval_start,
    "->",
    data_end
)

История для построения рекомендаций: 2015-05-03 03:00:04.384000 -> 2015-09-04 02:59:47.789000
Ranker train: 2015-09-04 02:59:47.789000 -> 2015-09-11 02:59:47.789000
Evaluation: 2015-09-11 02:59:47.789000 -> 2015-09-18 02:59:47.789000


In [7]:
# ============================================================
# 4. ФОРМИРУЕМ ПЕРИОДЫ RANKER TRAIN И EVALUATION
# ============================================================

ranker_events = events_rank[
    (events_rank["event_time"] >= ranker_start)
    &
    (events_rank["event_time"] < eval_start)
].copy()


eval_events = events_rank[
    (events_rank["event_time"] >= eval_start)
    &
    (events_rank["event_time"] < data_end)
].copy()


print(
    "Ranker train:",
    ranker_events["event_time"].min(),
    "->",
    ranker_events["event_time"].max()
)

print(
    "Evaluation:",
    eval_events["event_time"].min(),
    "->",
    eval_events["event_time"].max()
)

print(
    "Ranker events:",
    len(ranker_events)
)

print(
    "Evaluation events:",
    len(eval_events)
)

Ranker train: 2015-09-04 02:59:55.643000 -> 2015-09-11 02:59:45.585000
Evaluation: 2015-09-11 02:59:50.408000 -> 2015-09-18 02:59:47.788000
Ranker events: 132186
Evaluation events: 114738


In [8]:
# ============================================================
# 5. ФОРМИРУЕМ TARGET
# ============================================================

POSITIVE_EVENTS = [
    "addtocart",
    "transaction",
]


positive_pairs = (
    ranker_events[
        ranker_events["event"].isin(
            POSITIVE_EVENTS
        )
    ][
        [
            "user_id",
            "item_id",
        ]
    ]
    .drop_duplicates()
    .copy()
)


positive_pairs["target"] = 1


print(
    "Positive user-item pairs:",
    len(positive_pairs)
)

display(
    positive_pairs.head()
)

Positive user-item pairs: 3125


,user_id,item_id,target
1200121,685118,264476,1
1200165,237571,421931,1
1200177,981375,315939,1
1200222,685118,286268,1
1202317,703583,138454,1


In [9]:
import duckdb

positive_pairs_count = len(positive_pairs)

positive_found = duckdb.sql("""
    SELECT COUNT(*)
    FROM positive_pairs p
    INNER JOIN read_parquet(
        'candidates_result.parquet'
    ) c
        ON p.user_id = c.user_id
       AND p.item_id = c.item_id
""").fetchone()[0]

coverage = (
    positive_found / positive_pairs_count
    if positive_pairs_count > 0
    else 0
)

print(
    "Всего positive pairs:",
    positive_pairs_count
)

print(
    "Positive найдено среди candidates:",
    positive_found
)

print(
    "Coverage:",
    coverage
)

Всего positive pairs: 3125
Positive найдено среди candidates: 126
Coverage: 0.04032


In [10]:
FEATURES = [
    "als_score",
    "als_rank",
    "similarity_score",
    "similarity_rank",
    "popular_score",
    "popular_rank",
]

In [11]:
candidates.head(1)

,user_id,item_id,als_score,als_rank,popular_score,popular_rank,similarity_score,similarity_rank,similarity_sources,from_als,from_popular,from_similar,popular_score_log
0,1228945,450082,0.00211,44.0,33.0,51.0,NaN,NaN,NaN,1,1,0,3.526361


In [12]:
missing_features = [
    feature
    for feature in FEATURES
    if feature not in candidates.columns
]

if missing_features:
    raise ValueError(
        f"В candidates отсутствуют признаки: {missing_features}"
    )

In [13]:
# ============================================================
# СОЗДАЁМ ОБУЧАЮЩУЮ ТАБЛИЦУ RANKER
# ============================================================

ranker_users = positive_pairs[
    "user_id"
].unique()


# Сначала сокращаем огромный candidates
ranker_train = candidates[
    candidates["user_id"].isin(
        ranker_users
    )
].copy()


# Добавляем target
ranker_train = ranker_train.merge(
    positive_pairs,
    on=[
        "user_id",
        "item_id",
    ],
    how="left",
    validate="many_to_one",
)


ranker_train["target"] = (
    ranker_train["target"]
    .fillna(0)
    .astype("int8")
)


print(
    "Ranker rows:",
    len(ranker_train)
)

print(
    "Positive:",
    ranker_train["target"].sum()
)

print(
    "Positive share:",
    ranker_train["target"].mean()
)

Ranker rows: 80462
Positive: 126
Positive share: 0.001565956600631354


In [14]:

group_stats = (
    ranker_train
    .groupby("user_id")["target"]
    .agg(
        positives="sum",
        total="count"
    )
)





group_stats["negatives"] = (
    group_stats["total"]
    - group_stats["positives"]
)


valid_users = group_stats[
    (group_stats["positives"] > 0)
    &
    (group_stats["negatives"] > 0)
].index


ranker_train = ranker_train[
    ranker_train["user_id"].isin(
        valid_users
    )
].copy()




print(
    "Пользователей:",
    ranker_train["user_id"].nunique()
)

print(
    "Строк:",
    len(ranker_train)
)

print(
    "Доля target=1:",
    ranker_train["target"].mean()
)

Пользователей: 95
Строк: 29494
Доля target=1: 0.0042720553332881266


In [15]:
from catboost import CatBoostRanker


# CatBoost требует,
# чтобы строки одной группы шли подряд.
ranker_train = (
    ranker_train
    .sort_values(
        [
            "user_id",
            "als_rank"
        ]
    )
    .reset_index(drop=True)
)


ranker = CatBoostRanker(
    iterations=50,
    depth=6,
    learning_rate=0.05,
    loss_function="YetiRank",
    random_seed=42,
    verbose=1
)


ranker.fit(
    ranker_train[FEATURES],
    ranker_train["target"],
    group_id=ranker_train["user_id"]
)

0:	total: 106ms	remaining: 5.18s
1:	total: 152ms	remaining: 3.65s
2:	total: 206ms	remaining: 3.23s
3:	total: 295ms	remaining: 3.4s
4:	total: 403ms	remaining: 3.63s
5:	total: 495ms	remaining: 3.63s
6:	total: 579ms	remaining: 3.56s
7:	total: 630ms	remaining: 3.31s
8:	total: 678ms	remaining: 3.09s
9:	total: 727ms	remaining: 2.91s
10:	total: 773ms	remaining: 2.74s
11:	total: 820ms	remaining: 2.6s
12:	total: 867ms	remaining: 2.47s
13:	total: 915ms	remaining: 2.35s
14:	total: 963ms	remaining: 2.25s
15:	total: 1.01s	remaining: 2.15s
16:	total: 1.07s	remaining: 2.08s
17:	total: 1.12s	remaining: 1.99s
18:	total: 1.16s	remaining: 1.9s
19:	total: 1.21s	remaining: 1.81s
20:	total: 1.25s	remaining: 1.73s
21:	total: 1.3s	remaining: 1.66s
22:	total: 1.35s	remaining: 1.59s
23:	total: 1.42s	remaining: 1.54s
24:	total: 1.52s	remaining: 1.52s
25:	total: 1.59s	remaining: 1.47s
26:	total: 1.67s	remaining: 1.42s
27:	total: 1.73s	remaining: 1.36s
28:	total: 1.77s	remaining: 1.28s
29:	total: 1.81s	remaining: 

In [16]:
os.makedirs(
    "models",
    exist_ok=True
)

ranker.save_model(
    "models/catboost_ranker.cbm"
)

In [17]:
s3.upload_file(
    "models/catboost_ranker.cbm",
    bucket_name,
    "recsys/models/catboost_ranker.cbm"
)

In [18]:
s3.upload_file(
    "recommendations.parquet",
    bucket_name,
    "recsys/recommendations/recommendations.parquet"
)

In [19]:
import pyarrow as pa
import pyarrow.parquet as pq

BATCH_SIZE = 1_000_000

writer = None

try:

    for start in range(
        0,
        len(candidates),
        BATCH_SIZE
    ):

        end = min(
            start + BATCH_SIZE,
            len(candidates)
        )

        batch = candidates.iloc[
            start:end
        ]

        scores = (
            ranker.predict(
                batch[FEATURES]
            )
            .astype(np.float32)
        )

        result = batch[
            [
                "user_id",
                "item_id",
            ]
        ].copy()

        result["score"] = scores

        table = pa.Table.from_pandas(
            result,
            preserve_index=False
        )

        if writer is None:
            writer = pq.ParquetWriter(
                "recommendations_scored.parquet",
                table.schema,
                compression="snappy"
            )

        writer.write_table(table)

        print(
            f"Обработано: {end:,} / {len(candidates):,} "
            f"({end / len(candidates) * 100:.2f}%)"
        )

        del batch
        del result
        del scores
        del table

finally:

    if writer is not None:
        writer.close()

Обработано: 1,000,000 / 217,593,361 (0.46%)
Обработано: 2,000,000 / 217,593,361 (0.92%)
Обработано: 3,000,000 / 217,593,361 (1.38%)
Обработано: 4,000,000 / 217,593,361 (1.84%)
Обработано: 5,000,000 / 217,593,361 (2.30%)
Обработано: 6,000,000 / 217,593,361 (2.76%)
Обработано: 7,000,000 / 217,593,361 (3.22%)
Обработано: 8,000,000 / 217,593,361 (3.68%)
Обработано: 9,000,000 / 217,593,361 (4.14%)
Обработано: 10,000,000 / 217,593,361 (4.60%)
Обработано: 11,000,000 / 217,593,361 (5.06%)
Обработано: 12,000,000 / 217,593,361 (5.51%)
Обработано: 13,000,000 / 217,593,361 (5.97%)
Обработано: 14,000,000 / 217,593,361 (6.43%)
Обработано: 15,000,000 / 217,593,361 (6.89%)
Обработано: 16,000,000 / 217,593,361 (7.35%)
Обработано: 17,000,000 / 217,593,361 (7.81%)
Обработано: 18,000,000 / 217,593,361 (8.27%)
Обработано: 19,000,000 / 217,593,361 (8.73%)
Обработано: 20,000,000 / 217,593,361 (9.19%)
Обработано: 21,000,000 / 217,593,361 (9.65%)
Обработано: 22,000,000 / 217,593,361 (10.11%)
Обработано: 23,000

In [20]:
import duckdb

duckdb.sql("""
COPY (
    SELECT
        user_id,
        item_id,
        score,

        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY score DESC
        )::SMALLINT AS rank

    FROM read_parquet(
        'recommendations_scored.parquet'
    )
)
TO 'recommendations.parquet'
(
    FORMAT PARQUET,
    COMPRESSION SNAPPY,
    ROW_GROUP_SIZE 250000
)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [21]:
import duckdb

duckdb.sql("""
COPY (
    SELECT
        user_id,
        item_id,
        score,

        CAST(
            ROW_NUMBER() OVER (
                PARTITION BY user_id
                ORDER BY score DESC
            )
            AS SMALLINT
        ) AS rank

    FROM read_parquet(
        'recommendations_scored.parquet'
    )
)
TO 'recommendations.parquet'
(
    FORMAT PARQUET,
    COMPRESSION SNAPPY,
    ROW_GROUP_SIZE 250000
)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [22]:
recommendations = pd.read_parquet(
    "recommendations.parquet"
)

display(
    recommendations.head(20)
)

,user_id,item_id,score,rank
0,585,346673,1.213419,1
1,585,27127,1.189303,2
2,585,351044,1.105373,3
3,585,103156,1.009995,4
4,585,363794,0.998291,5
5,585,145403,0.941325,6
6,585,425758,0.935550,7
7,585,449571,0.935550,8
8,585,20981,0.902608,9
9,585,329166,0.776334,10


In [23]:


s3_key = (
    "recsys/recommendations/"
    "recommendations.parquet"
)

s3.upload_file(
    "recommendations.parquet",
    bucket_name,
    s3_key,
)

print(
    f"Загружено: "
    f"s3://{bucket_name}/{s3_key}"
)


print(
    "recommendations.parquet сохранён и загружен в S3"
)

Загружено: s3://s3-student-mle-20260525-e31877c13e-freetrack/recsys/recommendations/recommendations.parquet
recommendations.parquet сохранён и загружен в S3


In [24]:
import mlflow
import mlflow.catboost

os.environ[
    "MLFLOW_S3_ENDPOINT_URL"
] = "https://storage.yandexcloud.net"
mlflow.set_tracking_uri(
    "http://localhost:5000"
)

mlflow.set_experiment(
    "recsys_ranker"
)


with mlflow.start_run(
    run_name="catboost_ranker"
):

    # -----------------------------
    # PARAMS
    # -----------------------------

    mlflow.log_params({
        "iterations": 50,
        "depth": 6,
        "learning_rate": 0.05,
        "loss_function": "YetiRank",
        "random_seed": 42,

        "evaluation_days": 7,
        "ranker_days": 7,

        "features": ",".join(FEATURES),
        "positive_events":
            ",".join(POSITIVE_EVENTS),
    })


    # -----------------------------
    # METRICS
    # -----------------------------

    mlflow.log_metrics({

        "positive_pairs_count":
            int(positive_pairs_count),

        "positive_found":
            int(positive_found),

        "candidate_positive_coverage":
            float(coverage),

        "ranker_rows":
            int(len(ranker_train)),

        "ranker_users":
            int(
                ranker_train[
                    "user_id"
                ].nunique()
            ),

        "positive_count":
            int(
                ranker_train[
                    "target"
                ].sum()
            ),

        "positive_share":
            float(
                ranker_train[
                    "target"
                ].mean()
            ),
    })


    # -----------------------------
    # MODEL
    # -----------------------------

    mlflow.catboost.log_model(
        cb_model=ranker,
        artifact_path="model",
    )

2026/09/30 08:31:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run catboost_ranker at: http://localhost:5000/#/experiments/2/runs/3a58079d758942fbbaa7b79097aef714
🧪 View experiment at: http://localhost:5000/#/experiments/2


In [25]:
%reset -f

import gc
gc.collect()

0